# Воспроизведение baseline SASRec (Yandex YAMBDA)

Цель ноутбука — прогнать **оригинальный** код Яндекса (`yambda/benchmarks/models/sasrec/`) на Yambda-50M в режиме `listens` и убедиться, что воспроизводится цифра из таблицы `Listen+`:

- **NDCG@10 ≈ 0.0744**, **Recall@10 ≈ 0.0322** (см. `yandex_results/listen_results.png`).

Эта цифра — наш потолок для per-user скорера (gSASRec). Если наша реализация заметно ниже — значит дело в нашей тренировке/конфиге, а не в данных.

Запускать в **Google Colab (A100)**. Локально не имеет смысла — 100 эпох слишком долго.

## 1. Подтягиваем код Яндекса и ставим зависимости

Код бенчмарков лежит прямо в HF-датасете `yandex/yambda` под `benchmarks/` ([официальный README](https://huggingface.co/datasets/yandex/yambda/blob/main/benchmarks/README.md)). Качаем только нужную папку через `huggingface_hub.snapshot_download`.

In [1]:
!pip install -q polars click datasets huggingface_hub tqdm

In [2]:
from huggingface_hub import snapshot_download
import os

# Качаем только папку benchmarks/ из dataset-репозитория.
local_dir = snapshot_download(
    repo_id='yandex/yambda',
    repo_type='dataset',
    allow_patterns='benchmarks/*',
    local_dir='/content/yambda',
)
print('downloaded to:', local_dir)
print(os.listdir('/content/yambda/benchmarks'))

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

downloaded to: /content/yambda
['scripts', 'yambda', 'Dockerfile', 'models', 'pyproject.toml', 'README.md', 'tests', 'lint.sh', '.gitignore']


In [3]:
%cd /content/yambda/benchmarks
# Ставим пакет 'yambda' (модули timesplit и т.п.) в editable режиме.
# Если pyproject подтянет лишние зависимости (implicit, sansa) — игнорируем
# и используем PYTHONPATH как fallback.
!pip install -q -e . || echo 'pip install failed; will use PYTHONPATH fallback'

os.environ['PYTHONPATH'] = '/content/yambda/benchmarks:' + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH =', os.environ['PYTHONPATH'])

/content/yambda/benchmarks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 35.0 MB/s eta 0:00:00
  Building editable for yambda (pyproject.toml) ... done
PYTHONPATH = /content/yambda/benchmarks:/env/python


## 2. Скачиваем sequential/50m/listens.parquet

`train.py` ожидает структуру `data_dir/sequential/<size>/<interaction>.parquet` ([train.py:100](https://github.com/yandex-research/yambda/blob/main/benchmarks/models/sasrec/train.py#L100)).

In [4]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs('/content/data/sequential/50m', exist_ok=True)
src = hf_hub_download(
    repo_id='yandex/yambda',
    filename='sequential/50m/listens.parquet',
    repo_type='dataset',
)
dst = '/content/data/sequential/50m/listens.parquet'
if not os.path.exists(dst):
    shutil.copy(src, dst)
print('size, MB:', os.path.getsize(dst) / 1024 / 1024)

sequential/50m/listens.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

size, MB: 406.50250339508057


## 3. Тренировка SASRec на Listen+

Дефолтный конфиг Яндекса:
- embedding_dim=64, num_heads=2, num_layers=2, dropout=0.0
- max_seq_len=200, batch_size=256, lr=1e-3
- **num_epochs=100**
- loss: plain BCE с 1 uniform-негативом на позицию

На A100 100 эпох на 50M ≈ 1.5–3 часа. Если сессия может оборваться — снизь `--num_epochs` до 30–50 как sanity-чек.

In [5]:
!pip uninstall -y polars
!pip install -q --force-reinstall --no-cache-dir 'polars>=1.20'

Found existing installation: polars 1.35.2
Uninstalling polars-1.35.2:
  Successfully uninstalled polars-1.35.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 235.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-polars-cu12 26.2.1 requires polars<1.36,>=1.30, but you have polars 1.40.1 which is incompatible.


In [1]:
%cd /content/yambda/benchmarks/models/sasrec
!python train.py \
    --exp_name listens_50m \
    --data_dir /content/data \
    --checkpoint_dir /content/checkpoints \
    --size 50m \
    --interaction listens \
    --num_epochs 100 \
    --batch_size 256 \
    --device cuda:0

/content/yambda/benchmarks/models/sasrec
[2026-05-11 09:37:03] [DEBUG]: Preprocessing data...
[2026-05-11 09:37:08] [DEBUG]: Preprocessing data has finished!
[2026-05-11 09:37:11] [DEBUG]: Start training...
[2026-05-11 09:37:11] [DEBUG]: Start epoch 1
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: pin_memory_device is deprecated, the current accelerator will be used as the device,ignore pin_memory_device='cuda'.
  super().__init__(loader)
[2026-05-11 09:37:12] [DEBUG]: Start epoch 2
[2026-05-11 09:37:13] [DEBUG]: Start epoch 3
[2026-05-11 09:37:14] [DEBUG]: Start epoch 4
[2026-05-11 09:37:14] [DEBUG]: Start epoch 5
[2026-05-11 09:37:15] [DEBUG]: Start epoch 6
[2026-05-11 09:37:15] [DEBUG]: Start epoch 7
[2026-05-11 09:37:16] [DEBUG]: Start epoch 8
[2026-05-11 09:37:16] [DEBUG]: Start epoch 9
[2026-05-11 09:37:17] [DEBUG]: Start epoch 10
[2026-05-11 09:37:18] [DEBUG]: Start epoch 11
[2026-05-11 09:37:18] [DEBUG]: Start epoch 12
[2026-05-11 09:3

In [2]:
import os, polars as pl, sys
from data import preprocess


sys.path.insert(0, '/content/yambda/benchmarks/models/sasrec')

p = '/content/data/sequential/50m/listens.parquet'
print('file size, GB:', os.path.getsize(p) / 1e9)

df_raw = pl.scan_parquet(p).collect(engine="streaming")
print('raw rows (users):', df_raw.height)
print('schema:', df_raw.schema)
print(df_raw.head(2))

data = preprocess(pl.scan_parquet(p), 'listens', val_size=0, max_seq_len=200)
train_df = data.train.collect(engine="streaming")
print('num_items:', data.num_items)
print('train rows after preprocess:', train_df.height)
print('avg seq len:', train_df.select(pl.col('item_id').list.len().mean()).item())


file size, GB: 0.426248769
raw rows (users): 9238
schema: Schema({'uid': UInt32, 'timestamp': List(UInt32), 'item_id': List(UInt32), 'is_organic': List(UInt8), 'played_ratio_pct': List(UInt16), 'track_length_seconds': List(UInt32)})
shape: (2, 6)
┌─────┬─────────────────────┬──────────────┬─────────────┬────────────────────┬────────────────────┐
│ uid ┆ timestamp           ┆ item_id      ┆ is_organic  ┆ played_ratio_pct   ┆ track_length_secon │
│ --- ┆ ---                 ┆ ---          ┆ ---         ┆ ---                ┆ ds                 │
│ u32 ┆ list[u32]           ┆ list[u32]    ┆ list[u8]    ┆ list[u16]          ┆ ---                │
│     ┆                     ┆              ┆             ┆                    ┆ list[u32]          │
╞═════╪═════════════════════╪══════════════╪═════════════╪════════════════════╪════════════════════╡
│ 100 ┆ [39420, 39420, …    ┆ [8326270,    ┆ [0, 0, … 0] ┆ [100, 100, … 100]  ┆ [170, 105, … 165]  │
│     ┆ 25966140]           ┆ 1441281, …   ┆  

In [3]:
# eval.py хардкодит путь ./checkpoints/{exp_name}_best_state.pth и не принимает --checkpoint_dir.
# Делаем симлинк, чтобы не копировать большой файл.

sasrec_dir = '/content/yambda/benchmarks/models/sasrec'
os.makedirs(f'{sasrec_dir}/checkpoints', exist_ok=True)
link = f'{sasrec_dir}/checkpoints/listens_50m_best_state.pth'
src = '/content/checkpoints/listens_50m_best_state.pth'
if not os.path.exists(link):
    os.symlink(src, link)

print('checkpoint linked:', os.path.exists(link))

checkpoint linked: True


## 4. Оценка (NDCG@10, Recall@10)

Финальные метрики печатает `eval.py`. Ожидаем числа в районе строки `SASRec` в таблице Listen+ для Yambda-50M:

| Метрика | Ожидание |
|---|---|
| NDCG@10 | 0.0744 |
| NDCG@100 | 0.0764 |
| Recall@10 | 0.0322 |
| Recall@100 | 0.1028 |

In [4]:
%cd /content/yambda/benchmarks/models/sasrec
!python eval.py \
    --exp_name listens_50m \
    --data_dir /content/data \
    --size 50m \
    --interaction listens \
    --device cuda:0

/content/yambda/benchmarks/models/sasrec
[2026-05-11 09:45:56] [DEBUG]: Preprocessing data...
[2026-05-11 09:46:00] [DEBUG]: Preprocessing data has finished!
Calc topk by batches: 100% 36/36 [00:00<00:00, 447.73it/s]
Making target mask: 100% 4599/4599 [00:00<00:00, 12361.15it/s]
{'recall': {10: 0.06221185252070427, 50: 0.07666192203760147, 100: 0.10392861813306808}, 'ndcg': {10: 0.07418879866600037, 50: 0.07065223157405853, 100: 0.0804196149110794}, 'coverage': {10: 0.012979315503546729, 50: 0.023403972082585846, 100: 0.031050516320023328}}


## 5. Сохраняем чекпоинт в Google Drive (опционально)

Чтобы не терять обученный baseline между сессиями.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
out = '/content/drive/MyDrive/thesis/artifacts/yandex_sasrec_baseline'
os.makedirs(out, exist_ok=True)
shutil.copy('/content/checkpoints/listens_50m_best_state.pth', out)
print('saved to', out)

## Заметки

1. `train.py` сохраняет **последний** state, а не best-by-val ([train.py:144-145](https://github.com/yandex-research/yambda/blob/main/benchmarks/models/sasrec/train.py#L144-L145)). Промежуточные метрики не логирует — увидим только финальный `eval.py`.
2. Если хочется увидеть кривую обучения — нужно патчить `train.py` (вставить вызов `eval` после каждой эпохи). Для baseline-репродукции это излишне.
3. Запускали этот ноутбук — впиши в `docs/PHASE_1_LOG.md` фактические числа NDCG/Recall и расхождение с таблицей.